In [1]:
import os 
import openai
from langsmith import Client
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

/Users/kishorkumarparoi/Desktop/Maven - The AI Engineering Bootcamp /Resources/AIE5-main/21_Rag_Qdrant_Pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


In [3]:
client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

In [4]:
dataset = client.read_dataset(dataset_name="rag-evaluation-dataset")

In [5]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG system performance on question answering tasks based on product reviews.', data_type=<DataType.kv: 'kv'>, id=UUID('64bf9bd5-c777-4426-a769-577f772001cf'), created_at=datetime.datetime(2026, 5, 21, 5, 14, 13, 360607, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 5, 21, 5, 14, 13, 360607, tzinfo=TzInfo(0)), example_count=10, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.5-arm64-arm-64bit', 'sdk_version': '0.8.4', 'runtime_version': '3.12.5', 'langchain_version': '1.3.0', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.0'}})

In [6]:
list(client.list_examples(dataset_name="rag-evaluation-dataset"))

[<class 'langsmith.schemas.Example'>(id=53eb2865-982d-44a8-b72e-cbad0c94cf38, dataset_id=64bf9bd5-c777-4426-a769-577f772001cf, link='https://smith.langchain.com/o/564c6bbb-b517-4743-abef-e72d41dd8800/datasets/64bf9bd5-c777-4426-a769-577f772001cf/e/53eb2865-982d-44a8-b72e-cbad0c94cf38'),
 <class 'langsmith.schemas.Example'>(id=926f38c0-54e4-4692-8ad2-5158b87da621, dataset_id=64bf9bd5-c777-4426-a769-577f772001cf, link='https://smith.langchain.com/o/564c6bbb-b517-4743-abef-e72d41dd8800/datasets/64bf9bd5-c777-4426-a769-577f772001cf/e/926f38c0-54e4-4692-8ad2-5158b87da621'),
 <class 'langsmith.schemas.Example'>(id=50f3760f-a80c-4a6e-be48-f0b665690a34, dataset_id=64bf9bd5-c777-4426-a769-577f772001cf, link='https://smith.langchain.com/o/564c6bbb-b517-4743-abef-e72d41dd8800/datasets/64bf9bd5-c777-4426-a769-577f772001cf/e/50f3760f-a80c-4a6e-be48-f0b665690a34'),
 <class 'langsmith.schemas.Example'>(id=9a85b903-047c-46b4-8135-dc94ba5e0913, dataset_id=64bf9bd5-c777-4426-a769-577f772001cf, link='htt

In [7]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].inputs

In [8]:
reference_input

{'question': 'Which product specifies it is compatible only with the Garmin Vivofit 4 and not with Vivofit 1/2/3?'}

In [9]:
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs
reference_output

{'answer': "QGHXO Band for Garmin Vivofit 4 (id 6) — the description and features state it's custom designed for Garmin Vivofit 4 only and not for Vivofit 1/2/3."}

Rag Pipeline

In [18]:

from langsmith import traceable, get_current_run_tree
import openai
from qdrant_client import QdrantClient
import os

import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

qdrant_client = QdrantClient(
    url="http://localhost:6333",
)

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    current_run = get_current_run_tree()
    if current_run and getattr(response, "usage", None):
        try:
            current_run.add_metadata({
                "usage_metadata": {
                    "input_tokens": response.usage.prompt_tokens,
                    "total_tokens": response.usage.total_tokens,
                    "embedding_model": model,
                }
            })
        except Exception:
            logger = __import__("logging").getLogger(__name__)
            logger.debug("Failed to add embedding usage metadata to run")
    return response.data[0].embedding


def retrieve_data(query, qdrant_client=qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Data_Collection",
        query=query_embedding,
        limit=k,
        with_payload=True,
    )

    print("results.points: ", results.points)

    retrieved_context_ids = []
    retrieve_context = []
    similarity_scores = []
    retrieved_context_rating_numbers = []
    retrieve_context_main_categories = []
    retrieve_context_prices = []
    retrieve_context_images = []
    retrieve_context_videos = []
    retrieve_context_stores = []
    retrieve_context_categories = []
    retrieve_context_details = []
    retrieve_context_descriptions = []
    retrieve_context_features = []

    for result in results.points:
        payload = result.payload or {}
        retrieved_context_ids.append(result.id)
        similarity_scores.append(result.score)
        retrieve_context.append(payload.get('text', ''))
        retrieved_context_rating_numbers.append(payload.get('rating_number', None))
        retrieve_context_main_categories.append(payload.get('main_category', None))
        retrieve_context_prices.append(payload.get('price', None))
        retrieve_context_images.append(payload.get('images', None))
        retrieve_context_videos.append(payload.get('videos', None))
        retrieve_context_stores.append(payload.get('store', None))
        retrieve_context_categories.append(payload.get('categories', None))
        retrieve_context_details.append(payload.get('details', None))
        retrieve_context_descriptions.append(payload.get('description', None))
        retrieve_context_features.append(payload.get('features', None))

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieve_context': retrieve_context,
        'similarity_scores': similarity_scores,
        'retrieved_context_rating_numbers': retrieved_context_rating_numbers,
        'retrieve_context_main_categories': retrieve_context_main_categories,
        'retrieve_context_prices': retrieve_context_prices,
        'retrieve_context_images': retrieve_context_images,
        'retrieve_context_videos': retrieve_context_videos,
        'retrieve_context_stores': retrieve_context_stores,
        'retrieve_context_categories': retrieve_context_categories,
        'retrieve_context_details': retrieve_context_details,
        'retrieve_context_descriptions': retrieve_context_descriptions,
        'retrieve_context_features': retrieve_context_features,
    }

def process_context(context):
    formatted_context = []
    count = len(context.get('retrieved_context_ids', []))

    for index in range(count):
        point_id = context['retrieved_context_ids'][index]
        text = context['retrieve_context'][index] if index < len(context.get('retrieve_context', [])) else ''
        score = context['similarity_scores'][index] if index < len(context.get('similarity_scores', [])) else None
        rating_number = context['retrieved_context_rating_numbers'][index] if index < len(context.get('retrieved_context_rating_numbers', [])) else None
        main_category = context['retrieve_context_main_categories'][index] if index < len(context.get('retrieve_context_main_categories', [])) else ''
        price = context['retrieve_context_prices'][index] if index < len(context.get('retrieve_context_prices', [])) else None
        images = context['retrieve_context_images'][index] if index < len(context.get('retrieve_context_images', [])) else []
        videos = context['retrieve_context_videos'][index] if index < len(context.get('retrieve_context_videos', [])) else []
        store = context['retrieve_context_stores'][index] if index < len(context.get('retrieve_context_stores', [])) else ''
        categories = context['retrieve_context_categories'][index] if index < len(context.get('retrieve_context_categories', [])) else []
        details = context['retrieve_context_details'][index] if index < len(context.get('retrieve_context_details', [])) else ''
        description = context['retrieve_context_descriptions'][index] if index < len(context.get('retrieve_context_descriptions', [])) else ''
        features = context['retrieve_context_features'][index] if index < len(context.get('retrieve_context_features', [])) else []

        formatted_context.append(
            f"ID: {point_id}\n"
            f"Score: {score}\n"
            f"Rating Number: {rating_number}\n"
            f"Main Category: {main_category}\n"
            f"Price: {price}\n"
            f"Store: {store}\n"
            f"Categories: {categories}\n"
            f"Details: {details}\n"
            f"Description: {description}\n"
            f"Features: {features}\n"
            f"Images: {images}\n"
            f"Videos: {videos}\n"
            f"Text: {text}\n---"
        )

    return "\n".join(formatted_context)

@traceable(
        name="build_prompt",
        tags=["prompt_construction"],
        run_type="prompt"
)
def build_prompt(preprocessed_context, question):
    prompt = f"""You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      {preprocessed_context}
      Question: {question}
      Answer:"""
    return prompt

@traceable(
        name="gen_answer",
        tags=["answer_generation", "openai"],
        run_type="llm",
        metadata={"model": "gpt-5-nano", "ls-provider": "openai"},
)
def gen_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}]
    )

    current_run = get_current_run_tree()
    if current_run:
        try:
            current_run.add_metadata({
                "usage_metadata": {
                    "input_tokens": response.usage.prompt_tokens,
                    "output_tokens": response.usage.completion_tokens,
                    "total_tokens": response.usage.total_tokens,
                    "generation_model": "gpt-5-nano",
                }
            })
        except Exception:
            logger = __import__("logging").getLogger(__name__)
            logger.debug("Failed to add generation usage metadata to run")
    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
    retrieve_context = retrieve_data(question, qdrant_client=qdrant_client, k=top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = gen_answer(prompt)

    final_result = {
        "question": question,
        "answer": answer,
        "retrieved_context": retrieve_context,
    }

    return final_result


OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


Ragas Metrics

In [19]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_11332/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_11332/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_11332/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [20]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_11332/3421941080.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_11332/3421941080.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [21]:
reference_input

{'question': 'Which product specifies it is compatible only with the Garmin Vivofit 4 and not with Vivofit 1/2/3?'}

In [22]:
reference_output

{'answer': "QGHXO Band for Garmin Vivofit 4 (id 6) — the description and features state it's custom designed for Garmin Vivofit 4 only and not for Vivofit 1/2/3."}

In [23]:
result = rag_pipeline(reference_input["question"])

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_11332/4280455504.py:199: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)


results.points:  [ScoredPoint(id=6, version=17, score=0.5307127, payload={'product_id': '23332f63-9539-48ae-ae8e-4ce44521d3f0', 'text': 'QGHXO Band for Garmin Vivofit 4, Soft Silicone Replacement Watch Band Strap for Garmin Vivofit 4 Activity Tracker, Small, Large, Ten Colors (5PCS Bands-Girl, Large)', 'description': ['Compatibility', 'Custom designed for your precious', 'Garmin Vivofit 4', 'Activity Tracker, this Garmin Watch Sport Band features a combination of functionality and style. Fit for', 'Garmin Vivofit 4', 'Activity Tracker ONLY. NOT for Garmin Vivofit 1/Garmin Vivofit 2/Garmin Vivofit 3.', 'Feature', 'Material: Silicone. NOTE: Replacement Bands Only! Small fits wrists with a circumference of 122-188mm.  Large fits wrists with a circumference of 148-215mm. Models for selection: For Garmin Vivofit 4 Activity Tracker Only. Contracted design style, with you life contracted and not simple.', 'Package Included', 'Soft Silicone Replacement Watch Band Strap for Garmin Vivofit 4 Act

In [24]:
result

{'question': 'Which product specifies it is compatible only with the Garmin Vivofit 4 and not with Vivofit 1/2/3?',
 'answer': 'The QGHXO Band for Garmin Vivofit 4 (Soft Silicone Replacement Watch Band Strap for Garmin Vivofit 4 Activity Tracker, 5PCS Bands-Girl) specifies compatibility only with the Garmin Vivofit 4 and is NOT for Garmin Vivofit 1/2/3.',
 'retrieved_context': {'retrieved_context_ids': [6, 904, 3, 1791, 1559],
  'retrieve_context': ['QGHXO Band for Garmin Vivofit 4, Soft Silicone Replacement Watch Band Strap for Garmin Vivofit 4 Activity Tracker, Small, Large, Ten Colors (5PCS Bands-Girl, Large)',
   'C2D JOY for Garmin Vivofit 4 Case Leather Bands - Metal Steel Case with Leather Bands Only for Garmin Vivofit 4 Red (5.9-8.2in)',
   'NotoCity Compatible with Vivoactive 4 band 22mm Quick Release Silicone Bands/Garmin Darth Vader/First Avenger/Polar Vantage Smartwatch Sport Breathable Strap Replacement for Gear S3 Classic Watchband',
   'OVERSTEP Compatible with Garmin Fe

In [25]:
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]["retrieve_context"],
    )
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)

In [26]:
await ragas_faithfulness(result, '')

0.3333333333333333

In [27]:
async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]["retrieve_context"],
    )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    return await scorer.single_turn_ascore(sample)

In [28]:
await ragas_response_relevancy(result, '')

np.float64(0.3921162540499404)

In [29]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context"]["retrieved_context_ids"],
        reference_context_ids=example.get("retrieved_context_ids", [])
    )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [30]:
await ragas_context_precision_id_based(result, reference_output)

0.0